# Day 7 - Anagrams: counting and canonical keys

**Big idea:** two strings are anagrams when they have the same letters in the same amounts. Either *count* the letters, or *sort* them - both turn "is this a rearrangement?" into a simple equality check.

The code below is imported straight from `days/day-007/`, so these are my actual solutions.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

sys.path.insert(0, str(ROOT / "days" / "day-007"))
from valid_anagram import is_anagram, is_anagram_sorted
from group_anagrams import group_anagrams

## 1. Valid Anagram (LeetCode 242)

**Step 0, the one-line check:** different lengths -> can't be anagrams. Free, and it catches a lot.

**Approach A - counting:** count each character of `s`, then walk `t` and subtract. If a count would go below zero, it's not an anagram. Like checking a shopping bag against a receipt, item by item.

**Approach B - sorting:** sort both strings. Anagrams become identical: `"listen"` and `"silent"` both become `"eilnst"`.

In [2]:
for s, t in [("anagram", "nagaram"), ("rat", "car"), ("aacc", "ccac"), ("", ""), ("a", "a")]:
    print(f"{s!r:>10} vs {t!r:<10} counting={is_anagram(s, t)!s:<5}  sorting={is_anagram_sorted(s, t)}")

print("\nsorted('listen') =", "".join(sorted("listen")), "  sorted('silent') =", "".join(sorted("silent")))

 'anagram' vs 'nagaram'  counting=True   sorting=True
     'rat' vs 'car'      counting=False  sorting=False
    'aacc' vs 'ccac'     counting=False  sorting=False
        '' vs ''         counting=True   sorting=True
       'a' vs 'a'        counting=True   sorting=True

sorted('listen') = eilnst   sorted('silent') = eilnst


`"aacc"` vs `"ccac"` is the sneaky one: same *letters*, different *amounts*. Only checking "does each letter exist?" would wrongly say True - that's the mistake I'd most likely make under pressure.

| Approach | Time | Space | Note |
|---|---|---|---|
| Counting | O(n) | O(k) -> O(1) for a-z | fastest |
| Sorting | O(n log n) | O(n) | shortest to write |

(n = string length, k = number of distinct characters.)

## 2. What if the input is any Unicode, not just a-z?

- A 26-slot array breaks. A **dict** works for any character (that's why I used one).
- Characters that *look* the same can be different under the hood. Normalize first:

In [3]:
import unicodedata

composed = "é"            # é as one code point
decomposed = "é"         # e + a combining accent - also looks like é
print("look the same, equal?", composed == decomposed, "  lengths:", len(composed), len(decomposed))

nfc = lambda x: unicodedata.normalize("NFC", x)
print("after NFC normalize:  ", nfc(composed) == nfc(decomposed))

print("casefold can change length:", repr("Straße"), len("Straße"), "->", repr("Straße".casefold()), len("Straße".casefold()))

look the same, equal? False   lengths: 1 2
after NFC normalize:   True
casefold can change length: 'Straße' 6 -> 'strasse' 7


So for Unicode: **normalize (NFC) -> casefold if case shouldn't matter -> *then* do the length check -> count with a dict.** Order matters, because normalizing can change the length.

## 3. Group Anagrams (LeetCode 49)

**The idea: a canonical key.** Give every word a "fingerprint" that all its anagrams share, then group words by fingerprint in a dict.

Two good fingerprints:
- **Sorted letters:** `"eat"`, `"tea"`, `"ate"` -> `"aet"`
- **Letter counts:** a tuple of 26 numbers (how many a's, b's, ... z's)

In [4]:
def count_key(word):
    counts = [0] * 26
    for ch in word:
        counts[ord(ch) - ord("a")] += 1
    return tuple(counts)

for w in ["eat", "tea", "tan", "nat", "bat"]:
    print(f"{w}:  sorted key = {''.join(sorted(w))!r:<7}  count key = {count_key(w)}")

print("\ngrouped:", group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

eat:  sorted key = 'aet'    count key = (1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0)
tea:  sorted key = 'aet'    count key = (1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0)
tan:  sorted key = 'ant'    count key = (1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0)
nat:  sorted key = 'ant'    count key = (1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0)
bat:  sorted key = 'abt'    count key = (1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0)

grouped: [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 4. The trap: a list can't be a dict key

In [5]:
try:
    groups = {}
    groups[[0] * 26] = ["eat"]
except TypeError as e:
    print("TypeError:", e)

groups = {tuple([0] * 26): ["eat"]}
print("tuple key works:", len(groups), "group")

TypeError: unhashable type: 'list'
tuple key works: 1 group


**Why?** A dict files each key by its hash and trusts that hash never to change. A list is **mutable** - you could change it after putting it in, and the dict would look in the wrong drawer. So Python refuses lists up front. A **tuple** can't change, so it's hashable. Fix: `tuple(counts)`.

**A sneakier mistake:** turning the counts into a string with no separator. Two different words can collide and nothing raises an error:

In [6]:
counts_1 = [1, 11]   # e.g. one 'a', eleven 'b'
counts_2 = [11, 1]   # eleven 'a', one 'b'

print("no separator:  ", "".join(map(str, counts_1)), "==", "".join(map(str, counts_2)), "-> silently merged!")
print("with separator:", "#".join(map(str, counts_1)), "!=", "#".join(map(str, counts_2)))

no separator:   111 == 111 -> silently merged!
with separator: 1#11 != 11#1


## 5. Which key is faster?

Let n = number of strings, k = length of the longest one.

| Key | Time | Space |
|---|---|---|
| Sorted letters | O(n · k log k) | O(n · k) |
| 26-count tuple | O(n · k) | O(n · k) |

On paper the count key wins. But let's actually race them:

In [7]:
import random
import string
import timeit

def by_sort(strs):
    g = {}
    for w in strs:
        g.setdefault("".join(sorted(w)), []).append(w)
    return list(g.values())

def by_count(strs):
    g = {}
    for w in strs:
        g.setdefault(count_key(w), []).append(w)
    return list(g.values())

random.seed(0)
for k in (5, 100, 1000):
    words = ["".join(random.choices(string.ascii_lowercase, k=k)) for _ in range(2000)]
    s = min(timeit.repeat(lambda: by_sort(words), number=3, repeat=3))
    c = min(timeit.repeat(lambda: by_count(words), number=3, repeat=3))
    winner = "sort" if s < c else "count"
    print(f"k={k:<5} sort-key {s:.3f}s   count-key {c:.3f}s   -> {winner} wins")

k=5     sort-key 0.002s   count-key 0.003s   -> sort wins


k=100   sort-key 0.038s   count-key 0.025s   -> count wins

k=1000  sort-key 0.493s   count-key 0.236s   -> count wins


For **short words, sorting wins** - `sorted()` runs in fast C code, while my counting loop is slow Python. For **long words, counting wins** - the `log k` finally matters. Big-O describes how cost *grows*, not who wins at every size.

## 6. Day 7 recall quiz - short answers

- **(Day 6) Why does a timer around `call_next` give the wrong latency on SSE?** `call_next` returns when the response *starts*; the body streams afterwards. Measured: timer 0.00 s for a 0.92 s stream.
- **(Day 6) Why must a logging failure never fail `/chat`, and how is it enforced?** Monitoring must never take down the feature. Insert runs after the response is fully sent, wrapped in `try/except`.
- **(Day 4) Two Sum - what's in the hash map, and why does one pass work?** Keys = numbers seen so far, values = their index. When you reach the second number of a pair, the first is already stored. Check *before* storing, so a number can't pair with itself.
- **(Day 4) Contains Duplicate - what do you trade with a hash set vs sorting?** Memory for speed: set = O(n) time + O(n) memory; sort = O(n log n) time + little extra memory (but it changes the input list, and it can't stop early).

## Recap

- Valid Anagram: length check first, then count with a dict (O(n)) or sort (O(n log n)).
- Unicode: normalize -> casefold -> length check -> count with a dict.
- Group Anagrams: group by a canonical key - sorted letters or a 26-count **tuple**.
- Dict keys must be hashable: tuple yes, list no. Joining counts with no separator silently collides.
- Asymptotically faster isn't always faster in practice - measure.